# Parte 2 – API de lluvias (temporada 2024)

**Pregunta del trabajo:** ¿El Estado declara emergencia donde más llueve?

En este notebook medimos cuánto llovió en la capital de cada departamento entre el **1 de enero y el 31 de mayo de 2024**, usando la API gratuita de Open-Meteo (geocodificación + archivo histórico).

Resultado final: `datos/lluvias_por_departamento.csv`

In [2]:
import io
import os
import re
import time
import unicodedata

import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0"}
INICIO = "2024-01-01"
FIN = "2024-05-31"

# Misma lista que en la Parte 1, para que los nombres coincidan en el cruce (Parte 3)
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]

def normalizar(texto):
    """Quita notas [1], paréntesis, tildes y mayúsculas para poder comparar nombres."""
    texto = str(texto)
    texto = re.sub(r"\[.*?\]|\(.*?\)", "", texto)
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", texto).strip().lower()

os.makedirs("datos", exist_ok=True)

## Punto 1 – Tabla de departamentos y capitales (Wikipedia)

Wikipedia rechaza `pd.read_html(url)` directo (error 403) porque el pedido no se identifica como navegador. Por eso primero descargamos la página con `requests` y un `User-Agent`, y luego le pasamos el HTML a `pd.read_html`.

La página tiene varias tablas, así que buscamos automáticamente la que tiene columnas de *departamento* y *capital*.

In [3]:
url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
respuesta = requests.get(url, headers=HEADERS, timeout=30)
print("Código de estado:", respuesta.status_code)

tablas = pd.read_html(io.StringIO(respuesta.text))
print("Tablas encontradas en la página:", len(tablas))

def aplanar(columnas):
    """Algunas tablas tienen encabezados de dos niveles; los unimos en uno solo."""
    return [" ".join(str(x) for x in c) if isinstance(c, tuple) else str(c) for c in columnas]

candidatas = []
for i, t in enumerate(tablas):
    cols = aplanar(t.columns)
    col_dep = next((c for c in cols if "departamento" in c.lower()), None)
    col_cap = next((c for c in cols if "capital" in c.lower()), None)
    if col_dep and col_cap:
        candidatas.append((i, len(t), col_dep, col_cap))

print("Tablas candidatas (índice, filas, col_departamento, col_capital):")
for c in candidatas:
    print(c)

Código de estado: 200
Tablas encontradas en la página: 6
Tablas candidatas (índice, filas, col_departamento, col_capital):
(1, 24, 'Departamento', 'Capital')
(2, 8, 'Departamento', 'Capital')


In [4]:
# Elegimos la candidata con más filas
i, n, col_dep, col_cap = max(candidatas, key=lambda x: x[1])
tabla = tablas[i].copy()
tabla.columns = aplanar(tabla.columns)

wiki = tabla[[col_dep, col_cap]].copy()
wiki.columns = ["departamento", "capital"]
print("Filas leídas de Wikipedia:", len(wiki))
wiki

Filas leídas de Wikipedia: 24


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## Punto 2 – Verificación: 25 filas y limpieza de la columna capital

Comparamos los nombres de Wikipedia con nuestra lista oficial de 25 departamentos (sin tildes ni mayúsculas, para que "Ancash" y "Áncash" cuenten como iguales). Así:
- dejamos los nombres **escritos igual que en la Parte 1** (importante para el cruce),
- vemos qué filas de Wikipedia sobran (por ejemplo, totales o "Lima Metropolitana"),
- vemos qué departamentos faltan.

In [ ]:
wiki["dep_norm"] = wiki["departamento"].apply(normalizar)
oficiales_norm = [normalizar(d) for d in departamentos]

filas = []
for dep in departamentos:
    coincide = wiki[wiki["dep_norm"] == normalizar(dep)]
    capital = coincide.iloc[0]["capital"] if len(coincide) > 0 else None
    filas.append({"departamento": dep, "capital_wiki": capital})
capitales = pd.DataFrame(filas)

sobran = wiki[~wiki["dep_norm"].isin(oficiales_norm)]["departamento"].tolist()
faltan = capitales[capitales["capital_wiki"].isna()]["departamento"].tolist()
print("Departamentos encontrados:", capitales["capital_wiki"].notna().sum(), "de 25")
print("Filas de Wikipedia que no son uno de los 25:", sobran)
print("Departamentos que faltan:", faltan)

In [ ]:
# Completar a mano los que falten según lo que imprimió la celda anterior.
# El Callao suele aparecer como "Provincia Constitucional del Callao" o no aparecer.
# Fuente: la capital del Callao es la ciudad del Callao (dato del INEI / Gobierno Regional del Callao).
agregados_manual = {"Callao": "Callao"}

for dep, cap in agregados_manual.items():
    falta = (capitales["departamento"] == dep) & (capitales["capital_wiki"].isna())
    if falta.any():
        capitales.loc[falta, "capital_wiki"] = cap
        print(f"Agregado manualmente: {dep} -> {cap}")

print("Departamentos sin capital:", capitales["capital_wiki"].isna().sum())

In [ ]:
# Limpieza de la capital: quitamos notas [1] y textos entre paréntesis como "(de facto)"
capitales["capital"] = (capitales["capital_wiki"].astype(str)
                        .str.replace(r"\[.*?\]|\(.*?\)", "", regex=True)
                        .str.replace(r"\s+", " ", regex=True)
                        .str.strip())

cambiadas = capitales[capitales["capital"] != capitales["capital_wiki"].astype(str).str.strip()]
print("Capitales que tenían texto extra:")
print(cambiadas[["departamento", "capital_wiki", "capital"]])

assert len(capitales) == 25, "La tabla no tiene 25 filas"
assert capitales["capital"].notna().all(), "Hay capitales vacías"
capitales[["departamento", "capital"]]

**Decisiones de esta parte** *(revisen que coincida con lo que imprimieron arriba)*

- **Departamentos faltantes:** *[completar: cuáles faltaron, p. ej. Callao]*. Lo agregamos a mano con capital **Callao**, que es la capital de la Provincia Constitucional del Callao.
- **Lima:** Wikipedia muestra la capital como "Huacho (de facto)". El texto entre paréntesis hace que la API no encuentre la ciudad, así que lo quitamos. Usamos **Huacho** porque es la sede del Gobierno Regional de Lima. *(Alternativa válida: usar la ciudad de Lima. Lo importante es explicar la decisión.)* Limitación: una sola ciudad no representa bien un departamento con costa y sierra.
- **Otras capitales con texto extra:** *[completar con lo que haya salido en la tabla "Capitales que tenían texto extra"]*.

## Punto 3 – Latitud y longitud con la API de geocodificación

La API puede devolver varias ciudades con el mismo nombre, dentro y fuera del Perú. Nuestro criterio para elegir el resultado:

1. Nos quedamos solo con los resultados que tienen `country_code == "PE"`.
2. Entre esos, elegimos el primero cuyo `admin1` contenga el nombre del departamento (comparando sin tildes, así "Departamento de Cusco", "Ancash" o "Provincia Constitucional del Callao" coinciden).
3. Si ninguno coincide, tomamos el primero del Perú y lo marcamos como **"revisar"**.

Hacemos 1 segundo de pausa entre pedidos.

In [ ]:
# Si alguna capital sale mal, pongan aquí otro nombre para buscarla (p. ej. {"Pasco": "Cerro de Pasco"})
busqueda_alternativa = {}

def geocodificar(nombre, departamento):
    params = {"name": nombre, "count": 10, "language": "es"}
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search", params=params, timeout=30)
    r.raise_for_status()
    resultados = r.json().get("results", [])
    peruanos = [x for x in resultados if x.get("country_code") == "PE"]
    for x in peruanos:
        if normalizar(departamento) in normalizar(x.get("admin1", "")):
            return x, "coincide admin1", len(resultados), len(peruanos)
    if peruanos:
        return peruanos[0], "primer PE (revisar)", len(resultados), len(peruanos)
    return None, "sin resultado en PE", len(resultados), 0

filas = []
for _, fila in capitales.iterrows():
    nombre = busqueda_alternativa.get(fila["departamento"], fila["capital"])
    elegido, criterio, n_total, n_pe = geocodificar(nombre, fila["departamento"])
    filas.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1_api": elegido.get("admin1") if elegido else None,
        "pais": elegido.get("country_code") if elegido else None,
        "latitud": elegido.get("latitude") if elegido else None,
        "longitud": elegido.get("longitude") if elegido else None,
        "resultados_api": n_total,
        "resultados_peru": n_pe,
        "criterio": criterio,
    })
    time.sleep(1)

geo = pd.DataFrame(filas)
pd.set_option("display.width", 200)
geo

In [ ]:
# Verificación: todas en el Perú y en el departamento correcto
geo["en_peru"] = geo["pais"] == "PE"
geo["departamento_ok"] = geo.apply(
    lambda f: normalizar(f["departamento"]) in normalizar(f["admin1_api"] or ""), axis=1)

print("Filas en el Perú:", geo["en_peru"].sum(), "de 25")
print("Filas en el departamento correcto:", geo["departamento_ok"].sum(), "de 25")
print("\nFilas a revisar:")
print(geo[~(geo["en_peru"] & geo["departamento_ok"])][["departamento", "capital", "admin1_api", "latitud", "longitud"]])

**Cómo elegimos el resultado correcto**

- Filtramos por `country_code == "PE"` para descartar ciudades homónimas de otros países.
- Comparamos `admin1` con el departamento sin tildes y buscando el nombre *dentro* del texto, porque la API no siempre lo escribe igual (*"Departamento de ..."*, *"Ancash"*, *"Provincia Constitucional del Callao"*).
- Revisamos la tabla fila por fila. *[Completar: si alguna fila salió en "Filas a revisar", cuál fue, por qué y cómo se corrigió (por ejemplo, con `busqueda_alternativa`). Si salieron las 25 correctas, decirlo.]*

*Nota para la bitácora:* tomar simplemente "el primer resultado" o "el primero del Perú" puede elegir una ciudad equivocada; por eso añadimos la comparación con `admin1`.

## Punto 4 – Lluvia diaria de la temporada (API histórica)

Para cada capital pedimos `precipitation_sum` diario del 1 de enero al 31 de mayo de 2024, en hora de Lima. Hacemos 1 segundo de pausa entre pedidos.

2024 es bisiesto, así que cada departamento debería tener **152 días** (31 + 29 + 31 + 30 + 31).

In [ ]:
lluvia_diaria = {}

for _, fila in geo.iterrows():
    params = {
        "latitude": fila["latitud"],
        "longitude": fila["longitud"],
        "start_date": INICIO,
        "end_date": FIN,
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
    r.raise_for_status()
    diario = r.json()["daily"]
    lluvia_diaria[fila["departamento"]] = pd.DataFrame({
        "fecha": diario["time"],
        "precipitacion_mm": diario["precipitation_sum"],
    })
    print(f'{fila["departamento"]:<15} {len(diario["time"])} días descargados')
    time.sleep(1)

print("\nDepartamentos descargados:", len(lluvia_diaria))
print("¿Todos tienen 152 días?", all(len(df) == 152 for df in lluvia_diaria.values()))

## Punto 5 – Lluvia total, días de lluvia fuerte y días sin dato

- `lluvia_total_mm`: suma de la lluvia diaria de la temporada.
- `dias_lluvia_fuerte`: número de días con **20 mm o más**.
- Contamos los días que llegaron como `None` (pandas los convierte en `NaN`).

In [ ]:
filas = []
for dep, df in lluvia_diaria.items():
    lluvia = pd.to_numeric(df["precipitacion_mm"], errors="coerce")
    filas.append({
        "departamento": dep,
        "dias_descargados": len(df),
        "dias_sin_dato": int(lluvia.isna().sum()),
        "lluvia_total_mm": round(lluvia.sum(), 1),          # sum() ignora los NaN
        "dias_lluvia_fuerte": int((lluvia >= 20).sum()),   # NaN >= 20 da False
    })

resumen = pd.DataFrame(filas)
print("Total de días sin dato en todos los departamentos:", resumen["dias_sin_dato"].sum())
resumen.sort_values("lluvia_total_mm", ascending=False)

**Días sin dato**

*[Completar con el número que salió arriba.]*

- Si es **0**: lo comprobamos y no hubo que hacer nada; la suma usa los 152 días de cada departamento.
- Si es **mayor que 0**: `sum()` ignora los `NaN` y la comparación `>= 20` los cuenta como `False`, así que esos días se tratan como si no hubieran aportado lluvia. Dejamos la columna `dias_sin_dato` para saber en qué departamentos el total puede estar subestimado.

## Punto 6 – Guardar `datos/lluvias_por_departamento.csv`

In [ ]:
final = geo[["departamento", "capital", "latitud", "longitud"]].merge(
    resumen[["departamento", "lluvia_total_mm", "dias_lluvia_fuerte"]],
    on="departamento", how="left")

assert len(final) == 25, "El archivo final no tiene 25 filas"
assert final["lluvia_total_mm"].notna().all(), "Hay departamentos sin lluvia calculada"

final.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")
print("Archivo guardado con", len(final), "filas y columnas:", list(final.columns))
pd.read_csv("datos/lluvias_por_departamento.csv")